# Home Credit Default Risk — Feature Engineering

I turn the questions from `01_eda.ipynb` into a single applicant-level table. I keep the original application columns, add only history summaries I can explain, and deliberately skip competition-style rolling windows, trends, target encoding, and automated feature synthesis.

In [1]:
from pathlib import Path
from collections.abc import MutableMapping, Sequence
import gc
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

ID = "SK_ID_CURR"
TARGET = "TARGET"
EMPLOYMENT_SENTINEL = 365243


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root")


ROOT = find_project_root()
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
required_files = [
    "application_train.csv", "application_test.csv", "bureau.csv", "bureau_balance.csv",
    "previous_application.csv", "installments_payments.csv", "POS_CASH_balance.csv",
    "credit_card_balance.csv",
]
missing_files = [name for name in required_files if not (DATA / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing inputs: {missing_files}")

## 1. Shared safety helpers

Ratios preserve `NaN` when a denominator is invalid. Each auxiliary aggregate is checked at applicant grain before it can be joined. Temporal ranges are collected during the same reads used for aggregation, avoiding a second pass over the large event tables.

In [2]:
TEMPORAL_RANGES = {}


def safe_divide(numerator, denominator, *, positive_denominator=False):
    valid = denominator.gt(0) if positive_denominator else denominator.ne(0)
    return (numerator / denominator.where(valid)).replace([np.inf, -np.inf], np.nan)


def combine_metric(state: MutableMapping[str, pd.Series], name: str, values: pd.Series, operation="sum"):
    if name not in state:
        state[name] = values
    elif operation == "sum":
        state[name] = state[name].add(values, fill_value=0)
    elif operation == "max":
        state[name] = pd.concat([state[name], values], axis=1).max(axis=1)
    else:
        raise ValueError(operation)


def record_historical_range(table, column, values):
    valid = values.dropna()
    observed = {"min": float(valid.min()), "max": float(valid.max()), "missing": int(values.isna().sum()), "positive": int(values.gt(0).sum())}
    key = f"{table}.{column}"
    if key in TEMPORAL_RANGES:
        prior = TEMPORAL_RANGES[key]
        observed = {
            "min": min(prior["min"], observed["min"]), "max": max(prior["max"], observed["max"]),
            "missing": prior["missing"] + observed["missing"], "positive": prior["positive"] + observed["positive"],
        }
    TEMPORAL_RANGES[key] = observed
    if observed["positive"]:
        raise AssertionError(f"{key} contains future-looking values")


def validate_aggregate(frame, name):
    if frame.index.name != ID or not frame.index.is_unique:
        raise AssertionError(f"{name} is not at applicant grain")
    if TARGET in frame or frame.columns.duplicated().any():
        raise AssertionError(f"Invalid columns in {name}")
    if any("SK_ID_" in column for column in frame.columns):
        raise AssertionError(f"An identifier was accidentally aggregated in {name}")
    return frame.replace([np.inf, -np.inf], np.nan)

## 2. Application features

These retain the EDA notebook's sentinel, age, affordability, household, and external-score summaries. They use no learned thresholds or target information.

In [3]:
APPLICATION_FEATURES = [
    "DAYS_EMPLOYED_ANOMALY", "DAYS_EMPLOYED_CLEAN", "AGE_YEARS", "EMPLOYED_YEARS",
    "INCOME_CREDIT_RATIO", "ANNUITY_CREDIT_RATIO", "CREDIT_GOODS_RATIO",
    "INCOME_PER_PERSON", "EXT_SOURCE_MEAN", "EXT_SOURCE_COUNT", "EXT_SOURCE_MISSING_COUNT",
]


def application_features(application):
    if TARGET in application:
        raise ValueError("TARGET must be removed before feature construction")
    result = application.copy()
    anomaly = result["DAYS_EMPLOYED"].eq(EMPLOYMENT_SENTINEL)
    result["DAYS_EMPLOYED_ANOMALY"] = anomaly.astype("int8")
    result["DAYS_EMPLOYED_CLEAN"] = result["DAYS_EMPLOYED"].mask(anomaly)
    result["AGE_YEARS"] = -result["DAYS_BIRTH"] / 365.25
    result["EMPLOYED_YEARS"] = -result["DAYS_EMPLOYED_CLEAN"] / 365.25
    result["INCOME_CREDIT_RATIO"] = safe_divide(result["AMT_INCOME_TOTAL"], result["AMT_CREDIT"])
    result["ANNUITY_CREDIT_RATIO"] = safe_divide(result["AMT_ANNUITY"], result["AMT_CREDIT"])
    result["CREDIT_GOODS_RATIO"] = safe_divide(result["AMT_CREDIT"], result["AMT_GOODS_PRICE"])
    result["INCOME_PER_PERSON"] = safe_divide(result["AMT_INCOME_TOTAL"], result["CNT_FAM_MEMBERS"])
    ext = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    result["EXT_SOURCE_MEAN"] = result[ext].mean(axis=1)
    result["EXT_SOURCE_COUNT"] = result[ext].notna().sum(axis=1).astype("int8")
    result["EXT_SOURCE_MISSING_COUNT"] = (3 - result["EXT_SOURCE_COUNT"]).astype("int8")
    return result

## 3. Bureau features

Question represented: how much credit history exists, what share is active or closed, how large are credit/debt/overdue balances, how recent is the history, and whether microloan exposure is repeated rather than merely present.

In [4]:
BUREAU_FEATURES = [
    "BUREAU_RECORD_COUNT", "HAS_BUREAU_HISTORY", "HAS_MICROLOAN",
    "BUREAU_ACTIVE_COUNT", "BUREAU_CLOSED_COUNT", "BUREAU_ACTIVE_SHARE", "BUREAU_CLOSED_SHARE",
    "BUREAU_CREDIT_SUM", "BUREAU_DEBT_SUM", "BUREAU_OVERDUE_SUM", "BUREAU_MAX_OVERDUE",
    "BUREAU_DEBT_CREDIT_RATIO", "BUREAU_DAYS_CREDIT_MEAN", "BUREAU_DAYS_CREDIT_MAX",
    "BUREAU_MICROLOAN_COUNT",
]


def aggregate_bureau(path):
    columns = [
        ID, "SK_ID_BUREAU", "CREDIT_ACTIVE", "CREDIT_TYPE", "DAYS_CREDIT",
        "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "AMT_CREDIT_MAX_OVERDUE",
    ]
    bureau = pd.read_csv(path, usecols=columns)
    record_historical_range("bureau", "DAYS_CREDIT", bureau["DAYS_CREDIT"])
    if not bureau["SK_ID_BUREAU"].is_unique:
        raise AssertionError("SK_ID_BUREAU must uniquely identify bureau rows")
    bureau["IS_ACTIVE"] = bureau["CREDIT_ACTIVE"].eq("Active").astype("int8")
    bureau["IS_CLOSED"] = bureau["CREDIT_ACTIVE"].eq("Closed").astype("int8")
    bureau["IS_MICROLOAN"] = bureau["CREDIT_TYPE"].eq("Microloan").astype("int8")
    grouped = bureau.groupby(ID, sort=True)
    result = pd.DataFrame(index=grouped.size().index)
    result["BUREAU_RECORD_COUNT"] = grouped.size().astype("int32")
    result["HAS_BUREAU_HISTORY"] = 1
    result["BUREAU_ACTIVE_COUNT"] = grouped["IS_ACTIVE"].sum()
    result["BUREAU_CLOSED_COUNT"] = grouped["IS_CLOSED"].sum()
    result["BUREAU_ACTIVE_SHARE"] = grouped["IS_ACTIVE"].mean()
    result["BUREAU_CLOSED_SHARE"] = grouped["IS_CLOSED"].mean()
    result["BUREAU_CREDIT_SUM"] = grouped["AMT_CREDIT_SUM"].sum(min_count=1)
    result["BUREAU_DEBT_SUM"] = grouped["AMT_CREDIT_SUM_DEBT"].sum(min_count=1)
    result["BUREAU_OVERDUE_SUM"] = grouped["AMT_CREDIT_SUM_OVERDUE"].sum(min_count=1)
    result["BUREAU_MAX_OVERDUE"] = grouped["AMT_CREDIT_MAX_OVERDUE"].max()
    result["BUREAU_DEBT_CREDIT_RATIO"] = safe_divide(result["BUREAU_DEBT_SUM"], result["BUREAU_CREDIT_SUM"])
    result["BUREAU_DAYS_CREDIT_MEAN"] = grouped["DAYS_CREDIT"].mean()
    result["BUREAU_DAYS_CREDIT_MAX"] = grouped["DAYS_CREDIT"].max()
    result["BUREAU_MICROLOAN_COUNT"] = grouped["IS_MICROLOAN"].sum()
    result["HAS_MICROLOAN"] = result["BUREAU_MICROLOAN_COUNT"].gt(0).astype("int8")
    keys = bureau[["SK_ID_BUREAU", ID]].copy()
    del bureau
    return validate_aggregate(result, "bureau"), keys

## 4. Bureau-balance features

`STATUS='0'` is current, `1`–`5` are increasing delinquency bands, `C` is closed, and `X` is unknown. Only `1`–`5` count as delinquent. Monthly rows are first aggregated to `SK_ID_BUREAU`, then linked to `SK_ID_CURR` and summarized again.

In [5]:
BUREAU_BAL_FEATURES = [
    "BUREAU_BAL_MONTH_COUNT", "HAS_BUREAU_BAL_HISTORY", "BUREAU_BAL_EVER_DELINQUENT",
    "BUREAU_BAL_DELINQUENT_SHARE", "BUREAU_BAL_WORST_STATUS",
]


def aggregate_bureau_balance(path, bureau_keys, chunksize=1_000_000):
    state = {}
    for chunk in pd.read_csv(
        path, usecols=["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"], chunksize=chunksize,
        dtype={"SK_ID_BUREAU": "int32", "MONTHS_BALANCE": "int16", "STATUS": "category"},
    ):
        record_historical_range("bureau_balance", "MONTHS_BALANCE", chunk["MONTHS_BALANCE"])
        severity = chunk["STATUS"].map({str(value): float(value) for value in range(6)}).astype("float32")
        behavior = pd.DataFrame({
            "SK_ID_BUREAU": chunk["SK_ID_BUREAU"], "MONTH_COUNT": 1,
            "DELINQUENT_COUNT": severity.between(1, 5).astype("int8"), "WORST_STATUS": severity,
        }).groupby("SK_ID_BUREAU", sort=True)
        combine_metric(state, "MONTH_COUNT", behavior["MONTH_COUNT"].sum())
        combine_metric(state, "DELINQUENT_COUNT", behavior["DELINQUENT_COUNT"].sum())
        combine_metric(state, "WORST_STATUS", behavior["WORST_STATUS"].max(), "max")
    loan = pd.DataFrame(state).sort_index()
    loan.index.name = "SK_ID_BUREAU"
    loan = loan.join(bureau_keys.set_index("SK_ID_BUREAU"), how="inner")
    grouped = loan.groupby(ID, sort=True)
    result = pd.DataFrame(index=grouped.size().index)
    result["BUREAU_BAL_MONTH_COUNT"] = grouped["MONTH_COUNT"].sum()
    result["HAS_BUREAU_BAL_HISTORY"] = 1
    applicant_delinquent = grouped["DELINQUENT_COUNT"].sum()
    result["BUREAU_BAL_EVER_DELINQUENT"] = applicant_delinquent.gt(0).astype("int8")
    result["BUREAU_BAL_DELINQUENT_SHARE"] = safe_divide(applicant_delinquent, result["BUREAU_BAL_MONTH_COUNT"])
    result["BUREAU_BAL_WORST_STATUS"] = grouped["WORST_STATUS"].max()
    del loan
    return validate_aggregate(result, "bureau_balance")

## 5. Previous-application features

Counts and shares distinguish one refusal in one application from one refusal in ten. Amount and decision-day summaries retain scale and recency without introducing product interactions or target-derived categories.

In [6]:
PREVIOUS_FEATURES = [
    "PREVIOUS_APPLICATION_RECORD_COUNT", "HAS_PREVIOUS_APPLICATION_HISTORY", "HAS_REFUSED_PRIOR",
    "PREV_APPROVED_COUNT", "PREV_REFUSED_COUNT", "PREV_APPROVED_SHARE", "PREV_REFUSED_SHARE",
    "PREV_APPLICATION_AMT_MEAN", "PREV_APPLICATION_AMT_MAX", "PREV_CREDIT_AMT_MEAN",
    "PREV_CREDIT_AMT_MAX", "PREV_APPLICATION_CREDIT_RATIO_MEAN",
    "PREV_DAYS_DECISION_MEAN", "PREV_DAYS_DECISION_MAX",
]


def aggregate_previous(path):
    columns = [ID, "NAME_CONTRACT_STATUS", "AMT_APPLICATION", "AMT_CREDIT", "DAYS_DECISION"]
    previous = pd.read_csv(path, usecols=columns)
    record_historical_range("previous_application", "DAYS_DECISION", previous["DAYS_DECISION"])
    previous["APPROVED"] = previous["NAME_CONTRACT_STATUS"].eq("Approved").astype("int8")
    previous["REFUSED"] = previous["NAME_CONTRACT_STATUS"].eq("Refused").astype("int8")
    previous["APPLICATION_CREDIT_RATIO"] = safe_divide(previous["AMT_APPLICATION"], previous["AMT_CREDIT"], positive_denominator=True)
    grouped = previous.groupby(ID, sort=True)
    result = pd.DataFrame(index=grouped.size().index)
    result["PREVIOUS_APPLICATION_RECORD_COUNT"] = grouped.size().astype("int32")
    result["HAS_PREVIOUS_APPLICATION_HISTORY"] = 1
    result["PREV_APPROVED_COUNT"] = grouped["APPROVED"].sum()
    result["PREV_REFUSED_COUNT"] = grouped["REFUSED"].sum()
    result["PREV_APPROVED_SHARE"] = grouped["APPROVED"].mean()
    result["PREV_REFUSED_SHARE"] = grouped["REFUSED"].mean()
    result["HAS_REFUSED_PRIOR"] = result["PREV_REFUSED_COUNT"].gt(0).astype("int8")
    result["PREV_APPLICATION_AMT_MEAN"] = grouped["AMT_APPLICATION"].mean()
    result["PREV_APPLICATION_AMT_MAX"] = grouped["AMT_APPLICATION"].max()
    result["PREV_CREDIT_AMT_MEAN"] = grouped["AMT_CREDIT"].mean()
    result["PREV_CREDIT_AMT_MAX"] = grouped["AMT_CREDIT"].max()
    result["PREV_APPLICATION_CREDIT_RATIO_MEAN"] = grouped["APPLICATION_CREDIT_RATIO"].mean()
    result["PREV_DAYS_DECISION_MEAN"] = grouped["DAYS_DECISION"].mean()
    result["PREV_DAYS_DECISION_MAX"] = grouped["DAYS_DECISION"].max()
    del previous
    return validate_aggregate(result, "previous_application")

## 6. Installment features

`DPD=max(DAYS_ENTRY_PAYMENT-DAYS_INSTALMENT,0)` measures days past due, while `DBD` measures early payment. This prevents early payments from canceling late payments in the applicant mean. Payment ratios require a positive scheduled amount; shortfall is `clip(1-ratio,0,1)`.

In [7]:
INSTALLMENT_FEATURES = [
    "INSTALLMENT_RECORD_COUNT", "HAS_INSTALLMENT_HISTORY", "INSTALLMENT_MEAN_DPD",
    "INSTALLMENT_MAX_DPD", "INSTALLMENT_MEAN_DBD", "INSTALLMENT_LATE_SHARE",
    "INSTALLMENT_MEAN_PAYMENT_RATIO", "INSTALLMENT_MEAN_SHORTFALL_RATIO",
    "INSTALLMENT_UNDERPAID_SHARE", "ANY_LATE_INSTALLMENT", "ANY_INSTALLMENT_SHORTFALL",
]


def aggregate_installments(path, chunksize=1_000_000):
    state = {}
    columns = [ID, "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"]
    for chunk in pd.read_csv(path, usecols=columns, chunksize=chunksize):
        record_historical_range("installments", "DAYS_INSTALMENT", chunk["DAYS_INSTALMENT"])
        record_historical_range("installments", "DAYS_ENTRY_PAYMENT", chunk["DAYS_ENTRY_PAYMENT"])
        delay = chunk["DAYS_ENTRY_PAYMENT"] - chunk["DAYS_INSTALMENT"]
        dpd = delay.clip(lower=0)
        dbd = (-delay).clip(lower=0)
        ratio = safe_divide(chunk["AMT_PAYMENT"], chunk["AMT_INSTALMENT"], positive_denominator=True)
        shortfall = (1 - ratio).clip(0, 1)
        behavior = pd.DataFrame({
            ID: chunk[ID], "RECORD_COUNT": 1, "DPD_N": dpd.notna().astype("int8"),
            "DPD_SUM": dpd.fillna(0), "DPD_MAX": dpd, "DBD_SUM": dbd.fillna(0),
            "LATE_COUNT": dpd.gt(0).astype("int8"), "RATIO_N": ratio.notna().astype("int8"),
            "RATIO_SUM": ratio.fillna(0), "SHORTFALL_SUM": shortfall.fillna(0),
            "UNDERPAID_COUNT": shortfall.gt(0).astype("int8"),
        }).groupby(ID, sort=True)
        for metric in ["RECORD_COUNT", "DPD_N", "DPD_SUM", "DBD_SUM", "LATE_COUNT", "RATIO_N", "RATIO_SUM", "SHORTFALL_SUM", "UNDERPAID_COUNT"]:
            combine_metric(state, metric, behavior[metric].sum())
        combine_metric(state, "DPD_MAX", behavior["DPD_MAX"].max(), "max")
    raw = pd.DataFrame(state).sort_index(); raw.index.name = ID
    result = pd.DataFrame(index=raw.index)
    result["INSTALLMENT_RECORD_COUNT"] = raw["RECORD_COUNT"].astype("int32")
    result["HAS_INSTALLMENT_HISTORY"] = 1
    result["INSTALLMENT_MEAN_DPD"] = safe_divide(raw["DPD_SUM"], raw["DPD_N"])
    result["INSTALLMENT_MAX_DPD"] = raw["DPD_MAX"]
    result["INSTALLMENT_MEAN_DBD"] = safe_divide(raw["DBD_SUM"], raw["DPD_N"])
    result["INSTALLMENT_LATE_SHARE"] = safe_divide(raw["LATE_COUNT"], raw["DPD_N"])
    result["INSTALLMENT_MEAN_PAYMENT_RATIO"] = safe_divide(raw["RATIO_SUM"], raw["RATIO_N"])
    result["INSTALLMENT_MEAN_SHORTFALL_RATIO"] = safe_divide(raw["SHORTFALL_SUM"], raw["RATIO_N"])
    result["INSTALLMENT_UNDERPAID_SHARE"] = safe_divide(raw["UNDERPAID_COUNT"], raw["RATIO_N"])
    result["ANY_LATE_INSTALLMENT"] = raw["LATE_COUNT"].gt(0).astype("int8")
    result["ANY_INSTALLMENT_SHORTFALL"] = raw["UNDERPAID_COUNT"].gt(0).astype("int8")
    return validate_aggregate(result, "installments")

## 7. POS/CASH features

This compact family contrasts the number/share of delinquent months with simple ever-delinquent flags and maximum severity.

In [8]:
POS_FEATURES = [
    "POS_RECORD_COUNT", "HAS_POS_HISTORY", "POS_DPD_MONTHS", "POS_DPD_MAX", "POS_DPD_SHARE",
    "POS_DPD_DEF_MONTHS", "POS_DPD_DEF_MAX", "POS_DPD_DEF_SHARE", "ANY_POS_DPD", "ANY_POS_DPD_DEF",
]


def aggregate_pos(path, chunksize=1_000_000):
    state = {}
    for chunk in pd.read_csv(path, usecols=[ID, "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"], chunksize=chunksize):
        record_historical_range("POS_CASH", "MONTHS_BALANCE", chunk["MONTHS_BALANCE"])
        behavior = pd.DataFrame({
            ID: chunk[ID], "RECORD_COUNT": 1, "DPD_MONTHS": chunk["SK_DPD"].gt(0).astype("int8"),
            "DPD_MAX": chunk["SK_DPD"], "DPD_DEF_MONTHS": chunk["SK_DPD_DEF"].gt(0).astype("int8"),
            "DPD_DEF_MAX": chunk["SK_DPD_DEF"],
        }).groupby(ID, sort=True)
        for metric in ["RECORD_COUNT", "DPD_MONTHS", "DPD_DEF_MONTHS"]:
            combine_metric(state, metric, behavior[metric].sum())
        for metric in ["DPD_MAX", "DPD_DEF_MAX"]:
            combine_metric(state, metric, behavior[metric].max(), "max")
    raw = pd.DataFrame(state).sort_index(); raw.index.name = ID
    result = pd.DataFrame(index=raw.index)
    result["POS_RECORD_COUNT"] = raw["RECORD_COUNT"].astype("int32")
    result["HAS_POS_HISTORY"] = 1
    result["POS_DPD_MONTHS"] = raw["DPD_MONTHS"].astype("int32")
    result["POS_DPD_MAX"] = raw["DPD_MAX"]
    result["POS_DPD_SHARE"] = safe_divide(raw["DPD_MONTHS"], raw["RECORD_COUNT"])
    result["POS_DPD_DEF_MONTHS"] = raw["DPD_DEF_MONTHS"].astype("int32")
    result["POS_DPD_DEF_MAX"] = raw["DPD_DEF_MAX"]
    result["POS_DPD_DEF_SHARE"] = safe_divide(raw["DPD_DEF_MONTHS"], raw["RECORD_COUNT"])
    result["ANY_POS_DPD"] = raw["DPD_MONTHS"].gt(0).astype("int8")
    result["ANY_POS_DPD_DEF"] = raw["DPD_DEF_MONTHS"].gt(0).astype("int8")
    return validate_aggregate(result, "POS_CASH")

## 8. Credit-card features

Utilization is `max(AMT_BALANCE,0) / AMT_CREDIT_LIMIT_ACTUAL` only for positive limits. Balance retains absolute scale, while utilization and over-limit share express burden relative to available credit.

In [9]:
CARD_FEATURES = [
    "CARD_RECORD_COUNT", "HAS_CARD_HISTORY", "CARD_BALANCE_MEAN", "CARD_UTIL_MEAN", "CARD_UTIL_MAX",
    "CARD_OVER_LIMIT_MONTHS", "CARD_OVER_LIMIT_SHARE", "CARD_DPD_MONTHS", "CARD_DPD_MAX",
    "CARD_DPD_DEF_MAX", "CARD_DPD_SHARE", "EVER_OVER_CARD_LIMIT", "ANY_CARD_DPD",
]


def aggregate_cards(path, chunksize=750_000):
    state = {}
    columns = [ID, "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "SK_DPD", "SK_DPD_DEF"]
    for chunk in pd.read_csv(path, usecols=columns, chunksize=chunksize):
        record_historical_range("credit_card", "MONTHS_BALANCE", chunk["MONTHS_BALANCE"])
        util = safe_divide(chunk["AMT_BALANCE"].clip(lower=0), chunk["AMT_CREDIT_LIMIT_ACTUAL"], positive_denominator=True)
        behavior = pd.DataFrame({
            ID: chunk[ID], "RECORD_COUNT": 1, "BALANCE_N": chunk["AMT_BALANCE"].notna().astype("int8"),
            "BALANCE_SUM": chunk["AMT_BALANCE"].fillna(0), "UTIL_N": util.notna().astype("int8"),
            "UTIL_SUM": util.fillna(0), "UTIL_MAX": util, "OVER_LIMIT_MONTHS": util.gt(1).astype("int8"),
            "DPD_MONTHS": chunk["SK_DPD"].gt(0).astype("int8"), "DPD_MAX": chunk["SK_DPD"],
            "DPD_DEF_MAX": chunk["SK_DPD_DEF"],
        }).groupby(ID, sort=True)
        for metric in ["RECORD_COUNT", "BALANCE_N", "BALANCE_SUM", "UTIL_N", "UTIL_SUM", "OVER_LIMIT_MONTHS", "DPD_MONTHS"]:
            combine_metric(state, metric, behavior[metric].sum())
        for metric in ["UTIL_MAX", "DPD_MAX", "DPD_DEF_MAX"]:
            combine_metric(state, metric, behavior[metric].max(), "max")
    raw = pd.DataFrame(state).sort_index(); raw.index.name = ID
    result = pd.DataFrame(index=raw.index)
    result["CARD_RECORD_COUNT"] = raw["RECORD_COUNT"].astype("int32")
    result["HAS_CARD_HISTORY"] = 1
    result["CARD_BALANCE_MEAN"] = safe_divide(raw["BALANCE_SUM"], raw["BALANCE_N"])
    result["CARD_UTIL_MEAN"] = safe_divide(raw["UTIL_SUM"], raw["UTIL_N"])
    result["CARD_UTIL_MAX"] = raw["UTIL_MAX"]
    result["CARD_OVER_LIMIT_MONTHS"] = raw["OVER_LIMIT_MONTHS"].astype("int32")
    result["CARD_OVER_LIMIT_SHARE"] = safe_divide(raw["OVER_LIMIT_MONTHS"], raw["UTIL_N"])
    result["CARD_DPD_MONTHS"] = raw["DPD_MONTHS"].astype("int32")
    result["CARD_DPD_MAX"] = raw["DPD_MAX"]
    result["CARD_DPD_DEF_MAX"] = raw["DPD_DEF_MAX"]
    result["CARD_DPD_SHARE"] = safe_divide(raw["DPD_MONTHS"], raw["RECORD_COUNT"])
    result["EVER_OVER_CARD_LIMIT"] = raw["OVER_LIMIT_MONTHS"].gt(0).astype("int8")
    result["ANY_CARD_DPD"] = raw["DPD_MONTHS"].gt(0).astype("int8")
    return validate_aggregate(result, "credit_card")

## 9. Feature inventory and cross-table ratios

Only two historical affordability ratios are added: total bureau debt and total bureau credit relative to current income.

In [10]:
CROSS_TABLE_FEATURES = ["BUREAU_DEBT_INCOME_RATIO", "BUREAU_CREDIT_INCOME_RATIO"]
FEATURE_GROUPS = {
    "Application": APPLICATION_FEATURES, "Bureau": BUREAU_FEATURES,
    "Bureau Balance": BUREAU_BAL_FEATURES, "Previous Application": PREVIOUS_FEATURES,
    "Installments": INSTALLMENT_FEATURES, "POS": POS_FEATURES,
    "Credit Card": CARD_FEATURES, "Cross-table": CROSS_TABLE_FEATURES,
}
ENGINEERED_FEATURES = [feature for group in FEATURE_GROUPS.values() for feature in group]
assert len(ENGINEERED_FEATURES) == len(set(ENGINEERED_FEATURES)) == 81
display(pd.DataFrame([
    {"Source": source, "Feature": feature} for source, features in FEATURE_GROUPS.items() for feature in features
]))

,Source,Feature
0,Application,DAYS_EMPLOYED_ANOMALY
1,Application,DAYS_EMPLOYED_CLEAN
2,Application,AGE_YEARS
3,Application,EMPLOYED_YEARS
4,Application,INCOME_CREDIT_RATIO
5,Application,ANNUITY_CREDIT_RATIO
6,Application,CREDIT_GOODS_RATIO
7,Application,INCOME_PER_PERSON
8,Application,EXT_SOURCE_MEAN
9,Application,EXT_SOURCE_COUNT


## 10. Merge

Every history table is reduced to one row per applicant before a validated one-to-one left join. Counts and binary flags become zero when no history exists; rates, amounts, and severity remain missing.

In [11]:
COUNT_COLUMNS = [
    "BUREAU_RECORD_COUNT", "BUREAU_ACTIVE_COUNT", "BUREAU_CLOSED_COUNT", "BUREAU_MICROLOAN_COUNT",
    "BUREAU_BAL_MONTH_COUNT", "PREVIOUS_APPLICATION_RECORD_COUNT", "PREV_APPROVED_COUNT", "PREV_REFUSED_COUNT",
    "INSTALLMENT_RECORD_COUNT", "POS_RECORD_COUNT", "POS_DPD_MONTHS", "POS_DPD_DEF_MONTHS",
    "CARD_RECORD_COUNT", "CARD_OVER_LIMIT_MONTHS", "CARD_DPD_MONTHS",
]
FLAG_COLUMNS = [
    "DAYS_EMPLOYED_ANOMALY", "HAS_BUREAU_HISTORY", "HAS_MICROLOAN", "HAS_BUREAU_BAL_HISTORY",
    "BUREAU_BAL_EVER_DELINQUENT", "HAS_PREVIOUS_APPLICATION_HISTORY", "HAS_REFUSED_PRIOR",
    "HAS_INSTALLMENT_HISTORY", "ANY_LATE_INSTALLMENT", "ANY_INSTALLMENT_SHORTFALL",
    "HAS_POS_HISTORY", "ANY_POS_DPD", "ANY_POS_DPD_DEF", "HAS_CARD_HISTORY",
    "EVER_OVER_CARD_LIMIT", "ANY_CARD_DPD",
]


def load_applications(data_dir):
    train = pd.read_csv(data_dir / "application_train.csv", low_memory=False)
    test = pd.read_csv(data_dir / "application_test.csv", low_memory=False)
    if TARGET not in train or TARGET in test:
        raise AssertionError("TARGET placement in raw application tables is invalid")
    target = train.pop(TARGET)
    if train.columns.tolist() != test.columns.tolist():
        raise AssertionError("Raw application schemas do not align")
    return train, target, test


def merge_aggregate(frame, aggregate, name):
    rows = len(frame)
    result = frame.merge(aggregate, left_on=ID, right_index=True, how="left", validate="one_to_one", sort=False)
    if len(result) != rows:
        raise AssertionError(f"{name} changed applicant count")
    return result


def build_tables(data_dir):
    TEMPORAL_RANGES.clear()
    train_raw, target, test_raw = load_applications(data_dir)
    original_feature_count = test_raw.shape[1]
    train, test = application_features(train_raw), application_features(test_raw)
    bureau, bureau_keys = aggregate_bureau(data_dir / "bureau.csv")
    aggregates = [
        ("Bureau", bureau),
        ("Bureau Balance", aggregate_bureau_balance(data_dir / "bureau_balance.csv", bureau_keys)),
        ("Previous Application", aggregate_previous(data_dir / "previous_application.csv")),
        ("Installments", aggregate_installments(data_dir / "installments_payments.csv")),
        ("POS", aggregate_pos(data_dir / "POS_CASH_balance.csv")),
        ("Credit Card", aggregate_cards(data_dir / "credit_card_balance.csv")),
    ]
    del bureau_keys
    for name, aggregate in aggregates:
        train = merge_aggregate(train, aggregate, name)
        test = merge_aggregate(test, aggregate, name)
    for frame in [train, test]:
        frame[COUNT_COLUMNS] = frame[COUNT_COLUMNS].fillna(0).astype("int32")
        frame[FLAG_COLUMNS] = frame[FLAG_COLUMNS].fillna(0).astype("int8")
        frame["BUREAU_DEBT_INCOME_RATIO"] = safe_divide(frame["BUREAU_DEBT_SUM"], frame["AMT_INCOME_TOTAL"])
        frame["BUREAU_CREDIT_INCOME_RATIO"] = safe_divide(frame["BUREAU_CREDIT_SUM"], frame["AMT_INCOME_TOTAL"])
        frame.replace([np.inf, -np.inf], np.nan, inplace=True)
    train.insert(1, TARGET, target.to_numpy())
    return train, test, original_feature_count

## 11. Validation

In [12]:
ORIGINAL_V1_FEATURES = {
    "DAYS_EMPLOYED_ANOMALY", "DAYS_EMPLOYED_CLEAN", "AGE_YEARS", "EMPLOYED_YEARS",
    "INCOME_CREDIT_RATIO", "ANNUITY_CREDIT_RATIO", "CREDIT_GOODS_RATIO", "INCOME_PER_PERSON",
    "EXT_SOURCE_MEAN", "EXT_SOURCE_COUNT", "EXT_SOURCE_MISSING_COUNT", "BUREAU_RECORD_COUNT",
    "HAS_BUREAU_HISTORY", "HAS_MICROLOAN", "PREVIOUS_APPLICATION_RECORD_COUNT",
    "HAS_PREVIOUS_APPLICATION_HISTORY", "HAS_REFUSED_PRIOR", "INSTALLMENT_RECORD_COUNT",
    "HAS_INSTALLMENT_HISTORY", "INSTALLMENT_MEAN_DELAY_DAYS", "INSTALLMENT_MAX_DELAY_DAYS",
    "INSTALLMENT_LATE_SHARE", "INSTALLMENT_MEAN_SHORTFALL_RATIO", "INSTALLMENT_UNDERPAID_SHARE",
    "ANY_LATE_INSTALLMENT", "ANY_INSTALLMENT_SHORTFALL", "POS_RECORD_COUNT", "HAS_POS_HISTORY",
    "POS_MONTHS", "POS_DPD_MONTHS", "POS_DPD_MAX", "POS_DPD_DEF_MONTHS", "POS_DPD_DEF_MAX",
    "POS_DPD_SHARE", "POS_DPD_DEF_SHARE", "ANY_POS_DPD", "ANY_POS_DPD_DEF", "CARD_RECORD_COUNT",
    "CARD_MONTHS", "CARD_UTIL_MEAN", "CARD_UTIL_MAX", "CARD_OVER_LIMIT_MONTHS",
    "CARD_OVER_LIMIT_SHARE", "CARD_DPD_MONTHS", "CARD_DPD_MAX", "CARD_DPD_DEF_MAX",
    "CARD_DPD_SHARE", "HAS_CARD_HISTORY", "EVER_OVER_CARD_LIMIT", "ANY_CARD_DPD",
}


def fingerprint(frame):
    digest = hashlib.sha256()
    digest.update("|".join(frame.columns).encode())
    digest.update("|".join(map(str, frame.dtypes)).encode())
    digest.update(pd.util.hash_pandas_object(frame, index=True).to_numpy().tobytes())
    return digest.hexdigest()


def assert_no_inf(frame):
    for column in frame.select_dtypes(include="number"):
        if np.isinf(frame[column].to_numpy(dtype="float64", na_value=np.nan)).any():
            raise AssertionError(f"Infinite values found in {column}")


def validate_final(train, test, expected_train_rows, expected_test_rows):
    assert len(train) == expected_train_rows and len(test) == expected_test_rows
    assert train[ID].is_unique and test[ID].is_unique
    assert set(train[ID]).isdisjoint(set(test[ID]))
    assert TARGET in train and TARGET not in test
    assert train.drop(columns=TARGET).columns.tolist() == test.columns.tolist()
    assert not train.columns.duplicated().any() and not test.columns.duplicated().any()
    assert all("TARGET" not in column.upper() for column in test.columns)
    assert not any("SK_ID_" in feature for feature in ENGINEERED_FEATURES)
    for frame in [train, test]:
        assert_no_inf(frame)
        assert frame["DAYS_EMPLOYED_ANOMALY"].equals(frame["DAYS_EMPLOYED"].eq(EMPLOYMENT_SENTINEL).astype("int8"))
        history_pairs = [
            ("BUREAU_RECORD_COUNT", "HAS_BUREAU_HISTORY"), ("BUREAU_BAL_MONTH_COUNT", "HAS_BUREAU_BAL_HISTORY"),
            ("PREVIOUS_APPLICATION_RECORD_COUNT", "HAS_PREVIOUS_APPLICATION_HISTORY"),
            ("INSTALLMENT_RECORD_COUNT", "HAS_INSTALLMENT_HISTORY"), ("POS_RECORD_COUNT", "HAS_POS_HISTORY"),
            ("CARD_RECORD_COUNT", "HAS_CARD_HISTORY"),
        ]
        for count, flag in history_pairs:
            assert frame[count].gt(0).astype("int8").equals(frame[flag])
        no_installments = frame["HAS_INSTALLMENT_HISTORY"].eq(0)
        no_pos = frame["HAS_POS_HISTORY"].eq(0)
        no_card = frame["HAS_CARD_HISTORY"].eq(0)
        no_bureau_balance = frame["HAS_BUREAU_BAL_HISTORY"].eq(0)
        assert frame.loc[no_installments, "INSTALLMENT_MEAN_DPD"].isna().all()
        assert frame.loc[no_pos, "POS_DPD_SHARE"].isna().all()
        assert frame.loc[no_card, "CARD_UTIL_MEAN"].isna().all()
        assert frame.loc[no_bureau_balance, "BUREAU_BAL_DELINQUENT_SHARE"].isna().all()
    return True

## 12. Build twice, save, and report

In [13]:
expected_train_rows = pd.read_csv(DATA / "application_train.csv", usecols=[ID]).shape[0]
expected_test_rows = pd.read_csv(DATA / "application_test.csv", usecols=[ID]).shape[0]
features_train, features_test, original_feature_count = build_tables(DATA)
validate_final(features_train, features_test, expected_train_rows, expected_test_rows)
first_fingerprints = fingerprint(features_train), fingerprint(features_test)

PROCESSED.mkdir(parents=True, exist_ok=True)
features_train.to_parquet(PROCESSED / "features_train_refined.parquet", index=False)
features_test.to_parquet(PROCESSED / "features_test_refined.parquet", index=False)

repeat_train, repeat_test, _ = build_tables(DATA)
assert (fingerprint(repeat_train), fingerprint(repeat_test)) == first_fingerprints
del repeat_train, repeat_test
gc.collect()

retained_v1 = sorted(set(ENGINEERED_FEATURES) & ORIGINAL_V1_FEATURES)
new_refined = sorted(set(ENGINEERED_FEATURES) - ORIGINAL_V1_FEATURES)
report = {
    "train_shape": list(features_train.shape), "test_shape": list(features_test.shape),
    "original_feature_count_excluding_target": original_feature_count,
    "existing_v1_engineered_feature_count": len(ORIGINAL_V1_FEATURES),
    "retained_v1_feature_count": len(retained_v1), "new_refined_feature_count": len(new_refined),
    "total_engineered_feature_count": len(ENGINEERED_FEATURES),
    "feature_counts_by_source": {source: len(features) for source, features in FEATURE_GROUPS.items()},
    "train_memory_mb": features_train.memory_usage(deep=True).sum() / 1024**2,
    "test_memory_mb": features_test.memory_usage(deep=True).sum() / 1024**2,
    "temporal_ranges": TEMPORAL_RANGES, "validation_passed": True, "deterministic_second_run": True,
}
(PROCESSED / "feature_build_refined_report.json").write_text(json.dumps(report, indent=2) + "\n")
display(pd.Series(report, name="value").to_frame())
display(pd.DataFrame({"train_nulls": features_train[ENGINEERED_FEATURES].isna().sum(), "test_nulls": features_test[ENGINEERED_FEATURES].isna().sum()}))

,value
train_shape,"[307511, 203]"
test_shape,"[48744, 202]"
original_feature_count_excluding_target,121
existing_v1_engineered_feature_count,50
retained_v1_feature_count,46
new_refined_feature_count,35
total_engineered_feature_count,81
feature_counts_by_source,"{'Application': 11, 'Bureau': 15, 'Bureau Bala..."
train_memory_mb,639.3009
test_memory_mb,100.9835


,train_nulls,test_nulls
DAYS_EMPLOYED_ANOMALY,0,0
DAYS_EMPLOYED_CLEAN,55374,9274
AGE_YEARS,0,0
EMPLOYED_YEARS,55374,9274
INCOME_CREDIT_RATIO,0,0
ANNUITY_CREDIT_RATIO,12,24
CREDIT_GOODS_RATIO,278,0
INCOME_PER_PERSON,2,0
EXT_SOURCE_MEAN,172,7
EXT_SOURCE_COUNT,0,0


## Feature summary

- **Application:** age, employment sentinel handling, affordability ratios, household-normalized income, and external-score availability.
- **Bureau:** history depth, active/closed composition, exposure, debt, overdue burden, recency, and a compact microloan count.
- **Bureau balance:** four behavior/depth summaries plus an explicit history flag, using the correct loan-to-applicant hierarchy.
- **Previous applications:** outcome counts/shares, requested and granted amounts, their ratio, and decision recency.
- **Installments:** non-negative DPD/DBD, payment completeness, shortfall, and interpretable event flags.
- **POS and cards:** compact frequency, severity, utilization, balance, and over-limit summaries.
- **Cross-table:** only bureau debt/income and credit/income.

Limitations: these are lifetime summaries, so changing behavior over time is intentionally not represented. `bureau_balance` treats only numeric STATUS 1–5 as delinquency. Missing historical severity remains `NaN`; downstream imputation and all assessment of predictive value belong inside fixed cross-validation folds.